In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

BASE_DIR = r"E:\Research\Paper_A_XAI_IDS"
RAW_DIR = os.path.join(BASE_DIR, "02_Data", "raw", "NSL-KDD")
PROCESSED_DIR = os.path.join(BASE_DIR, "02_Data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_path = os.path.join(RAW_DIR, "KDDTrain+.txt")
test_path = os.path.join(RAW_DIR, "KDDTest+.txt")

col_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

train_df = pd.read_csv(train_path, header=None, names=col_names)
test_df = pd.read_csv(test_path, header=None, names=col_names)

print(train_df.shape, test_df.shape)
train_df.head()

(125973, 43) (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [2]:
print(train_df["label"].value_counts().head(15))
print("\nTest labels:")
print(test_df["label"].value_counts().head(15))

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
Name: count, dtype: int64

Test labels:
label
normal           9711
neptune          4657
guess_passwd     1231
mscan             996
warezmaster       944
apache2           737
satan             735
processtable      685
smurf             665
back              359
snmpguess         331
saint             319
mailbomb          293
snmpgetattack     178
portsweep         157
Name: count, dtype: int64


In [3]:
attack_category_map = {
    "normal": "normal",
    "back": "dos", "land": "dos", "neptune": "dos", "pod": "dos", "smurf": "dos",
    "teardrop": "dos", "mailbomb": "dos", "apache2": "dos", "processtable": "dos",
    "udpstorm": "dos", "worm": "dos",
    "ipsweep": "probe", "nmap": "probe", "portsweep": "probe", "satan": "probe",
    "mscan": "probe", "saint": "probe",
    "ftp_write": "r2l", "guess_passwd": "r2l", "imap": "r2l", "multihop": "r2l",
    "phf": "r2l", "spy": "r2l", "warezclient": "r2l", "warezmaster": "r2l",
    "sendmail": "r2l", "named": "r2l", "snmpgetattack": "r2l", "snmpguess": "r2l",
    "xlock": "r2l", "xsnoop": "r2l", "httptunnel": "r2l",
    "buffer_overflow": "u2r", "loadmodule": "u2r", "perl": "u2r", "rootkit": "u2r",
    "ps": "u2r", "sqlattack": "u2r", "xterm": "u2r"
}

train_df["attack_category"] = train_df["label"].map(attack_category_map)
test_df["attack_category"] = test_df["label"].map(attack_category_map)

print(train_df["attack_category"].value_counts(dropna=False))
print(test_df["attack_category"].value_counts(dropna=False))

attack_category
normal    67343
dos       45927
probe     11656
r2l         995
u2r          52
Name: count, dtype: int64
attack_category
normal    9711
dos       7460
r2l       2885
probe     2421
u2r         67
Name: count, dtype: int64


In [4]:
train_df["binary_label"] = np.where(train_df["label"] == "normal", 0, 1)
test_df["binary_label"] = np.where(test_df["label"] == "normal", 0, 1)

print(train_df["binary_label"].value_counts())
print(test_df["binary_label"].value_counts())

binary_label
0    67343
1    58630
Name: count, dtype: int64
binary_label
1    12833
0     9711
Name: count, dtype: int64


In [5]:
label_encoder = LabelEncoder()
train_df["attack_category_encoded"] = label_encoder.fit_transform(train_df["attack_category"])
test_df["attack_category_encoded"] = label_encoder.transform(test_df["attack_category"])

print(list(label_encoder.classes_))

['dos', 'normal', 'probe', 'r2l', 'u2r']


In [6]:
feature_cols = [c for c in train_df.columns if c not in ["label", "difficulty", "attack_category", "binary_label", "attack_category_encoded"]]
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train_binary = train_df["binary_label"].copy()
y_test_binary = test_df["binary_label"].copy()

y_train_multi = train_df["attack_category_encoded"].copy()
y_test_multi = test_df["attack_category_encoded"].copy()

print(X_train.shape, X_test.shape)

(125973, 41) (22544, 41)


In [7]:
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

print("Categorical:", categorical_cols)
print("Numeric count:", len(numeric_cols))

Categorical: ['protocol_type', 'service', 'flag']
Numeric count: 38


In [8]:
X_all = pd.concat([X_train, X_test], axis=0, ignore_index=True)
X_all_encoded = pd.get_dummies(X_all, columns=categorical_cols, drop_first=False)

X_train_enc = X_all_encoded.iloc[:len(X_train)].copy()
X_test_enc = X_all_encoded.iloc[len(X_train):].copy()

print(X_train_enc.shape, X_test_enc.shape)

(125973, 122) (22544, 122)


In [9]:
print("Missing train encoded:", X_train_enc.isnull().sum().sum())
print("Missing test encoded :", X_test_enc.isnull().sum().sum())

Missing train encoded: 0
Missing test encoded : 0


In [10]:
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_enc, y_train_binary, test_size=0.2, random_state=42, stratify=y_train_binary
)

print(X_train_final.shape, X_val_final.shape)
print(y_train_final.value_counts(normalize=True))
print(y_val_final.value_counts(normalize=True))

(100778, 122) (25195, 122)
binary_label
0    0.534581
1    0.465419
Name: proportion, dtype: float64
binary_label
0    0.53459
1    0.46541
Name: proportion, dtype: float64


In [11]:
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train_final)
X_val_scaled = scaler.transform(X_val_final)
X_test_scaled = scaler.transform(X_test_enc)

print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

(100778, 122) (25195, 122) (22544, 122)


In [12]:
processed_train_path = os.path.join(PROCESSED_DIR, "X_train_final.csv")
processed_val_path = os.path.join(PROCESSED_DIR, "X_val_final.csv")
processed_test_path = os.path.join(PROCESSED_DIR, "X_test_final.csv")

pd.DataFrame(X_train_final).to_csv(processed_train_path, index=False)
pd.DataFrame(X_val_final).to_csv(processed_val_path, index=False)
pd.DataFrame(X_test_enc).to_csv(processed_test_path, index=False)

print("Saved:")
print(processed_train_path)
print(processed_val_path)
print(processed_test_path)

Saved:
E:\Research\Paper_A_XAI_IDS\02_Data\processed\X_train_final.csv
E:\Research\Paper_A_XAI_IDS\02_Data\processed\X_val_final.csv
E:\Research\Paper_A_XAI_IDS\02_Data\processed\X_test_final.csv


In [13]:
labels_train_path = os.path.join(PROCESSED_DIR, "y_train_binary.csv")
labels_val_path = os.path.join(PROCESSED_DIR, "y_val_binary.csv")
labels_test_path = os.path.join(PROCESSED_DIR, "y_test_binary.csv")

y_train_final.to_csv(labels_train_path, index=False)
y_val_final.to_csv(labels_val_path, index=False)
y_test_binary.to_csv(labels_test_path, index=False)

print("Saved label files.")

Saved label files.


In [14]:
summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(X_train_final), len(X_val_final), len(X_test_enc)],
    "columns": [X_train_final.shape[1], X_val_final.shape[1], X_test_enc.shape[1]]
})
summary_path = os.path.join(PROCESSED_DIR, "preprocessing_summary.csv")
summary.to_csv(summary_path, index=False)
summary

,split,rows,columns
0,train,100778,122
1,val,25195,122
2,test,22544,122


In [15]:
print("Preprocessing complete.")
print("Next step: create a baseline notebook named 03_nsl_kdd_baselines.ipynb")

Preprocessing complete.
Next step: create a baseline notebook named 03_nsl_kdd_baselines.ipynb
